# 04 - baselines: representation x model class

Is RNA-FM actually better than the preprint's 16 hand-crafted features? The answer depends on
the head you put on top, so this notebook fills in the full 2x2.

All four cells use **identical GroupKFold folds** and target `ln(Ts)`.

Hand-crafted features come from the May reproduction in `handcrafted_features/features_output-all.csv`.
Note those were computed before the pheA-1 / ECK120034435 annotation swap, so 2 of 582 rows
carry the uncorrected values there.

In [1]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.simplefilter("ignore")
REPO = Path.cwd().parent if Path.cwd().name == "probing" else Path.cwd()
EMB, RES = REPO / "probing/embeddings", REPO / "probing/results"
ALPHAS = np.logspace(-3, 6, 40)

FOLDS = json.loads((RES / "folds.json").read_text())
print("folds:", FOLDS["n_folds"], "| seed:", FOLDS["seed"], "| schemes:", list(FOLDS["schemes"]))

folds: 10 | seed: 42 | schemes: ['KFold', 'GroupKFold']


In [2]:
def per_fold(make_model, X, y, folds):
    """Fit on each training fold, score the held-out fold. Returns (r2 array, rho array)."""
    r2, rho = [], []
    for tr, te in folds:
        tr, te = np.array(tr), np.array(te)
        m = make_model()
        m.fit(X[tr], y[tr])
        p = m.predict(X[te])
        r2.append(r2_score(y[te], p))
        rho.append(spearmanr(y[te], p).statistic)
    return np.array(r2), np.array(rho)

RIDGE = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS))

def one_hot(seqs, L):
    idx = {c: i for i, c in enumerate("ACGU")}
    M = np.zeros((len(seqs), L, 4), dtype=np.float32)
    for i, s in enumerate(seqs):
        for j, c in enumerate(s):
            M[i, j, idx[c]] = 1.0
    return M.reshape(len(seqs), -1)

In [3]:
from xgboost import XGBRegressor

df = pd.read_csv(REPO/"probing/data/chen2013_window_up90.csv").rename(columns={"strength":"Ts"})
y  = np.log(df.Ts.to_numpy())

# best RNA-FM arm from notebook 02: layer 6, mean, 109 nt window
E6 = np.load(EMB/"rnafm_L109_layer6_tokens.npy")[:, 1:110, :]
RNAFM = E6.mean(1)

hc = pd.read_csv(REPO/"handcrafted_features/features_output-all.csv")
drop = ['Name','Average Strength','Orig_index','Utract','Atract','Stem','Hairpin','Loop','Structure']
HC = hc.set_index("Name").reindex(df.name)[[c for c in hc.columns if c not in drop]].to_numpy()
assert np.isfinite(HC).all(), "feature join failed"
print("RNA-FM", RNAFM.shape, "  hand-crafted", HC.shape)

RNA-FM (582, 640)   hand-crafted (582, 16)


In [4]:
# the preprint's tuned hyperparameters
TUNED = dict(n_estimators=4000, learning_rate=0.001, max_depth=5,
             min_child_weight=3, subsample=0.65, random_state=42)
XGB = lambda: XGBRegressor(**TUNED)
folds = FOLDS["schemes"]["GroupKFold"]

cells = {}
for rn, X in [("RNA-FM L6 mean", RNAFM), ("hand-crafted 16", HC)]:
    for mn, mk in [("Ridge", RIDGE), ("XGBoost", XGB)]:
        r, _ = per_fold(mk, X, y, folds)
        cells[(rn, mn)] = r
        print(f"  {rn:<20} + {mn:<8} R2 = {r.mean():+.3f} +/- {r.std():.3f}", flush=True)

  RNA-FM L6 mean       + Ridge    R2 = +0.634 +/- 0.131
  RNA-FM L6 mean       + XGBoost  R2 = +0.601 +/- 0.096
  hand-crafted 16      + Ridge    R2 = +0.429 +/- 0.203
  hand-crafted 16      + XGBoost  R2 = +0.620 +/- 0.121


In [9]:
print("=== 2x2: representation x model class, GroupKFold, ln(Ts) ===")
def ms(r):
    return f"{r.mean():+.3f} +/- {r.std():.3f}"   # per-fold R2 as mean +/- sd, as in the cell above
print(f"{'':<22}{'Ridge':>20}{'XGBoost':>20}")
for rn in ["RNA-FM L6 mean", "hand-crafted 16"]:
    print(f"{rn:<22}{ms(cells[(rn,'Ridge')]):>20}{ms(cells[(rn,'XGBoost')]):>20}")

print("\n=== paired tests (identical folds) ===")
def cmp(a, b, label):
    d = cells[a] - cells[b]
    print(f"{label:<50}{d.mean():>+8.3f}  wins {int((d>0).sum())}/10  p={wilcoxon(cells[a], cells[b]).pvalue:.3f}")
cmp(("RNA-FM L6 mean","Ridge"),   ("hand-crafted 16","XGBoost"), "RNA-FM+Ridge   - handcrafted+XGBoost")
cmp(("RNA-FM L6 mean","XGBoost"), ("hand-crafted 16","XGBoost"), "RNA-FM+XGBoost - handcrafted+XGBoost")
cmp(("RNA-FM L6 mean","XGBoost"), ("RNA-FM L6 mean","Ridge"), "RNA-FM+XGBoost - RNA-FM+Ridge")
cmp(("hand-crafted 16","XGBoost"),    ("hand-crafted 16","Ridge"),    "handcrafted+XGBoost - handcrafted+Ridge")
cmp(("RNA-FM L6 mean","Ridge"),    ("hand-crafted 16","Ridge"),    "RNA-FM+Ridge - handcrafted+Ridge")

=== 2x2: representation x model class, GroupKFold, ln(Ts) ===
                                     Ridge             XGBoost
RNA-FM L6 mean            +0.634 +/- 0.131    +0.601 +/- 0.096
hand-crafted 16           +0.429 +/- 0.203    +0.620 +/- 0.121

=== paired tests (identical folds) ===
RNA-FM+Ridge   - handcrafted+XGBoost                +0.014  wins 6/10  p=0.492
RNA-FM+XGBoost - handcrafted+XGBoost                -0.020  wins 4/10  p=0.492
RNA-FM+XGBoost - RNA-FM+Ridge                       -0.034  wins 3/10  p=0.084
handcrafted+XGBoost - handcrafted+Ridge             +0.191  wins 10/10  p=0.002
RNA-FM+Ridge - handcrafted+Ridge                    +0.205  wins 10/10  p=0.002


**Why this matters:** an earlier ridge-vs-ridge comparison made RNA-FM look far ahead, but the
hand-crafted features need a nonlinear head to work at all. Comparing them under the same head
is the fair test of the *representation*; comparing best-pipeline to best-pipeline is the fair
test of the *method*. They give different answers, so report which one you mean.